# Acts 1–2 — Agents, harnesses, and context

These are the lesson cells tested in Colab. Run this notebook from top to bottom; it also works independently of the other act notebooks.

**Setup:** use a Python 3.13+ kernel. The next cells find or clone the repository and install its pinned direct dependencies. If your Colab setup already did this, skip those two cells and run the imports and settings below from the repository directory.

All imports are grouped here, above the lessons. Set any model environment override before running the lesson imports.


In [ ]:
from pathlib import Path
from getpass import getpass
import tomllib
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

In [ ]:
import os
import json
from getpass import getpass
from pathlib import Path

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

from acts.act1_agent import build_agent
from acts.act2_context import (build_act2, preview_context, summarize_context_run,
                               print_context_comparison, print_shipping_demo, shipping_policy_demo)
from acts.act3_workflow import build_act3
from acts.act4_harness import build_act4
from acts.act5_skills import build_act5
from acts.act6_composition import build_act6
from cli import (
    print_agent_result,
    print_cart,
    print_workflow_result,
    print_harness_result,
    print_composition_result,
)
from formaggio.config import MODEL, load_json
from formaggio.agents.context import (
    customer_ask, customer_message, instructions, load_scenario,
)
from formaggio.agents.harness import Harness
from formaggio.agents.layers import Trace, Budget, Context, CartCheck
from formaggio.agents.runtime import FUNCTION_LIMITS, MODEL_OPTIONS
from formaggio.agents.tools import build_tools
from formaggio.operations.observability import Recorder
from formaggio.shop.data_models import AgentReply
from formaggio.shop.store import Store
from formaggio.operations.chat_view import show_chat


## Before we begin

We will build a cheese-shop assistant in six steps. The shop data and rules are fictional classroom examples.

The cells use live model calls by default. Set `USE_FIXTURES = True` to use scripted responses in Acts 3–6. Acts 1–2 still use the live model.

When a review is needed, you will see the cart or email draft and type `approve` or `decline`. Orders are simulated, and approved emails are saved as local HTML files; no purchases or emails are sent.


In [ ]:
SCENARIO = "standard"  # Act 1 baseline.
CONTEXT_SCENARIO = "personalized"  # Same request for both modes in Act 2.
CONTEXT_OVERRIDE_SCENARIO = "preference-override"  # Today overrides saved taste.
USE_FIXTURES = False  # Scripted responses in Acts 3–6 only; Acts 1–2 are live.
RUN_CONTEXT_EVALUATION = False  # Optional extra live calls at the end of Act 2.
CONTEXT_REPEATS = 3

DB_PATH = Path("outputs/colab.sqlite")
OUTPUT_ROOT = Path("outputs/colab_artifacts")
MEMORY_PATH = Path("outputs/colab_memory.sqlite")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def ask_for_decision():
    while True:
        decision = input("Type approve or decline: ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision
        print("Please enter approve or decline.")


async def review_order(ticket):
    print("\n👤 Manager review")
    print(customer_ask(ticket.request))
    print_cart(ticket.report)
    return ask_for_decision()


async def review_email(review):
    print("\n👤 Review this email draft")
    print(f"To: {review.draft.recipient}")
    print(f"Subject: {review.draft.subject}\n")
    print(review.draft.body)
    return ask_for_decision()


print(f"Model: {MODEL}")
print("Acts 3–6:", "scripted responses" if USE_FIXTURES else "live model")


### API key

If you already configured `OPENAI_API_KEY` in Colab, this cell reuses it. Otherwise it asks privately when live calls are enabled. The key is not printed or saved in the notebook.


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


## Act 1 — Build an agent

**Question: Can the agent use tools to propose a cart, and can the harness independently check what it produced?**

An agent combines a model, instructions, and tools. Here, the model can look up cheeses, check stock, calculate prices, and propose a tasting.

The SDK handles the loop: call the model, run its requested tools, and return the results until it gives a final answer.

First, run the agent directly. It can propose a cart, but it cannot place an order.


In [ ]:
store = Store()
request = load_scenario(SCENARIO)
print("👤 Customer ask:\n" + customer_ask(request))

async with AsyncOpenAI(timeout=45, max_retries=0) as api:
    client = OpenAIChatClient(
        model=MODEL,
        async_client=api,
        function_invocation_configuration=FUNCTION_LIMITS,
    )
    agent = Agent(
        client=client,
        name="FormaggioAssistant",
        instructions=instructions(store),
        tools=build_tools(store, request, None, None),
        default_options={**MODEL_OPTIONS, "response_format": AgentReply},
    )
    bare_response = await agent.run(customer_message(request))

print("\n💬 Agent's response:\n" + bare_response.value.message)


### Add checks around the agent

The harness runs the same agent definition with four named layers:

- **Trace:** show and record what happened.
- **Budget:** limit model calls, tool calls, and runtime.
- **Context:** choose the extra information supplied to the agent.
- **CartCheck:** independently check the final proposed cart.

`build_agent` contains the agent definition shown above, with hooks for these layers. Calling `.run(...)` starts a fresh run using the chosen scenario.

The cart check reports problems; it does not repair the cart or place an order.


In [ ]:
act1 = (
    Harness(build_agent, model=MODEL, scenario=SCENARIO)
    .add(Trace())
    .add(Budget(model_calls=8, tool_calls=20, seconds=120))
    .add(Context("basic"))
    .add(CartCheck())
)

with Recorder(DB_PATH) as recorder:
    act1_result = await act1.run(recorder, progress=print)

print_agent_result(act1_result)


### View the recorded conversation and checks

The left panel shows the recorded task, the final assistant answer when available, and application results. The right panel shows checks, review decisions, and expandable evidence. Internal model replies remain in the evidence panel.

This cell reads the saved run; it does not call the model or repeat any action. Using `DB_PATH` works after the run cell closes its recorder. Add `backstage=False` to show only the request, response, and application results.


In [ ]:
show_chat(DB_PATH, act1_result["run_id"])


## Act 2 — Give the agent useful context

**Question: What useful information did we add, and did the agent use it appropriately?**

Our returning customer asks for a tasting without repeating their saved preferences:
**mild cheeses and nonalcoholic pairings**. `CONTEXT_SCENARIO` selects this example.
The model, tools, instructions and current request stay the same between modes.
Basic gets no saved profile; enriched gets this customer's profile and a filtered shelf.

Look at the actual cheese choices. Our classroom rubric calls funk **0–2** mild.
Enriched should choose mild cheeses; basic is not graded on a preference it never saw.
Read the final pairing suggestions yourself: the automated cheese score does not grade
pairing prose. A tool listing wine as an option is not the same as recommending wine.

A matching answer is evidence, not proof of causation. Both modes may match, and enriched
may fail. Keep those results. Repeat runs to measure consistency. `standard` remains an
optional baseline whose saved preferences overlap the current request.


In [ ]:
context_preview = preview_context(CONTEXT_SCENARIO)

print("📚 What changes before the first model call?")
print("Basic: the same shop rules, customer request, and tools; no extra context sources.")
for item in context_preview["enriched"]["sources"]:
    print(f"Enriched adds {item['source']}: {item['reason']}")

print("\nSaved preferences:", context_preview["enriched"]["sources"][-1]["content"].get("preferences", []))

print("\n🔎 Excluded from the enriched starting list:")
reason_labels = {
    "fictional_shipping_policy": "raw milk: blocked by the fictional destination policy",
    "allergen_conflict": "conflicts with the confirmed allergies",
    "insufficient_stock_for_minimum": "not enough stock for the minimum serving",
}
for product in context_preview["enriched"]["excluded_products"]:
    reasons = "; ".join(reason_labels.get(reason, reason) for reason in product["reasons"])
    print(f"- {product['product_id']}: {reasons}")


### See the harness apply the shipping policy

Context can help the model avoid a bad choice. **Application code must still check it.**
This fixed counterexample contains Comté, a raw-milk cheese. The same cart is rejected
for PA and accepted for NY by `Store.validate`, used by both the CartCheck layer and checkout.
This is separate from the agent's cart and makes no model call or order.

Act 2 reports the violation. Act 3 will show the workflow refusing this proposal,
requesting a revision, and placing only the corrected cart. The PA restriction is a
**fictional classroom stand-in for regulatory policy**, not an actual statement of law.


In [ ]:
print_shipping_demo()
shipping_reports = shipping_policy_demo()


### Run both context modes

Now compare the same selected scenario in both modes.


In [ ]:
context_results = {}
context_comparison = []
for mode in ("basic", "enriched"):
    print(f"\n🧪 Running {mode} context")
    lesson = build_act2(scenario=CONTEXT_SCENARIO, mode=mode, model=MODEL)
    with Recorder(DB_PATH) as recorder:
        result = await lesson.run(recorder, progress=print)
        context_comparison.append(summarize_context_run(result, recorder.timeline(result["run_id"])))
    context_results[mode] = result
    print_agent_result(result)

print_context_comparison(list(context_results.values()))
print("\nCost of each run (tokens summed across all calls):")
for row in context_comparison:
    print(f"{row['mode']}: {row['model_calls']} model calls; {row['tool_calls']} tool calls; "
          f"input tokens {row['input_tokens']}; output tokens {row['output_tokens']}")
print("None means usage was not recorded. More context is not automatically better.")


### Today takes priority

The same customer now asks for a cheese at funk 4 or higher. Run enriched again.
The context check should show today's request wins over the saved mild preference.
This makes one additional live run.


In [ ]:
override_lesson = build_act2(scenario=CONTEXT_OVERRIDE_SCENARIO, mode="enriched", model=MODEL)
with Recorder(DB_PATH) as recorder:
    override_result = await override_lesson.run(recorder, progress=print)
print_agent_result(override_result)


### View the recorded conversation and checks

The left panel shows the recorded task, the final assistant answer when available, and application results. The right panel shows checks, review decisions, and expandable evidence. Internal model replies remain in the evidence panel.

This cell reads the saved run; it does not call the model or repeat any action. Using `DB_PATH` works after the run cell closes its recorder. Add `backstage=False` to show only the request, response, and application results.


In [ ]:
for mode, result in context_results.items():
    show_chat(DB_PATH, result["run_id"], title=f"Act 2 · {mode} context")


### Optional: repeat the context comparison

One run is an example, not a reliable comparison. Set `RUN_CONTEXT_EVALUATION = True`
in the shared settings, then run the next cell to try each mode `CONTEXT_REPEATS` times.
It makes **additional live model calls**, including when `USE_FIXTURES = True`.
Leave the switch off to skip it. Every trial starts with fresh shop inventory.

Compare the fraction of carts that pass, the average number of calls, and input tokens.
If both modes perform equally well, the experiment has not shown a benefit from adding
context for this task. A few trials are still a small sample; record mixed or failed
results rather than choosing only the best examples.


In [ ]:
context_trials = []
if RUN_CONTEXT_EVALUATION:
    if type(CONTEXT_REPEATS) is not int or CONTEXT_REPEATS < 2:
        raise ValueError("Use at least two repeats for a comparison.")
    for repetition in range(CONTEXT_REPEATS):
        for mode in ("basic", "enriched"):
            lesson = build_act2(scenario=CONTEXT_SCENARIO, mode=mode, model=MODEL)
            try:
                with Recorder(DB_PATH) as recorder:
                    trial = await lesson.run(recorder)
                    row = summarize_context_run(trial, recorder.timeline(trial["run_id"]))
                row["error"] = None
            except Exception as error:
                row = {"mode": mode, "cart_check": "error", "error": str(error)}
            row["repetition"] = repetition + 1
            context_trials.append(row)
            print(f"Trial {repetition + 1} · {mode}: {row['cart_check']}")
            if row["error"]:
                print("  Error:", row["error"])

    print("\n📊 Repeated comparison")
    for mode in ("basic", "enriched"):
        rows = [row for row in context_trials if row["mode"] == mode]
        completed = [row for row in rows if not row["error"]]
        passed = sum(row["cart_check"] == "passed" for row in rows)
        print(f"{mode}: {passed}/{len(rows)} carts passed; {len(rows) - len(completed)} run errors")
        personalized = sum(row.get("preference_check") == "pass" for row in rows)
        print(f"  Applicable context checks passed: {personalized}/{len(rows)} (basic receives no saved preference)")
        for metric in ("model_calls", "tool_calls", "input_tokens", "output_tokens"):
            measured = [row[metric] for row in completed if row[metric] is not None]
            average = f"{sum(measured) / len(measured):.1f}" if measured else "unavailable"
            print(f"  Average {metric}: {average} ({len(measured)}/{len(rows)} trials measured)")
else:
    print("Optional repeated comparison skipped. Enable it in the shared settings when ready.")


### Check the lesson automatically

From the repository terminal, run `python -m tests -p test_notebook.py` to execute the saved lesson cells with local scripted responses. The tests skip cloning, package installation, and credential entry; they use temporary output files and simulate review decisions. Run `python -m tests` for the full offline suite.
